In [9]:
import torch_geometric
print(torch_geometric.__version__) 

2.6.1


In [10]:
from ogb.graphproppred import PygGraphPropPredDataset
from torch_geometric.loader import DataLoader
import torch.serialization
import os

os.environ['OGB_URL'] = "https://mirrors.tuna.tsinghua.edu.cn/ogb/"

torch.serialization.add_safe_globals([
	torch_geometric.data.data.DataEdgeAttr,
	torch_geometric.data.data.DataTensorAttr,
	torch_geometric.data.storage.GlobalStorage
])

dataset = PygGraphPropPredDataset(name="ogbg-molhiv", root='dataset/')
split_idx = dataset.get_idx_split() 

train_loader = DataLoader(dataset[split_idx["train"]], batch_size=8, shuffle=True)
valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=8, shuffle=False)
test_loader = DataLoader(dataset[split_idx["test"]], batch_size=8, shuffle=False)


In [11]:
import numpy as np
from sklearn.metrics import roc_auc_score
import math
from tqdm import tqdm
import torch

def compute_metrics(scores, labels):
    if len(np.unique(labels)) < 2:
        return {'auc': np.nan, 'mrr': np.nan, 'hit5': np.nan}
    
    auc = roc_auc_score(labels, scores)
    
    sorted_indices = np.argsort(-scores)
    sorted_labels = labels[sorted_indices]
    pos_ranks = np.where(sorted_labels == 1)[0] + 1
    
    mrr = np.mean(1. / pos_ranks) if len(pos_ranks) > 0 else 0.0
    hit5 = np.mean(pos_ranks <= 5) if len(pos_ranks) > 0 else 0.0
    
    return {'auc': auc, 'mrr': mrr, 'hit5': hit5}

def compute_heuristics(data):
    edge_index = data.edge_index.cpu().numpy()
    num_nodes = data.num_nodes
    
    adj = [set() for _ in range(num_nodes)]
    for u, v in edge_index.T:
        adj[u].add(v)
        adj[v].add(u)
    
    edge_set = set()
    for u, v in edge_index.T:
        if u <= v:
            edge_set.add((u, v))
        else:
            edge_set.add((v, u))
    pos_samples = list(edge_set)
    num_pos = len(pos_samples)
    
    if num_pos == 0:
        return None
    
    neg_samples = []
    max_attempts = num_pos * 10
    attempts = 0
    
    while len(neg_samples) < num_pos and attempts < max_attempts:
        u, v = np.random.choice(num_nodes, 2, replace=False)
        if u > v:
            u, v = v, u
        if (u, v) not in edge_set:
            neg_samples.append((u, v))
        attempts += 1
    
    if len(neg_samples) < num_pos:
        return None
    
    scores_cn, scores_pa, scores_aa = [], [], []
    labels = []
    
    for u, v in pos_samples:
        common = adj[u].intersection(adj[v])
        scores_cn.append(len(common))
        scores_pa.append(len(adj[u]) * len(adj[v]))
        scores_aa.append(sum(1.0 / math.log(len(adj[z])) for z in common))
        labels.append(1)
    
    for u, v in neg_samples:
        common = adj[u].intersection(adj[v])
        scores_cn.append(len(common))
        scores_pa.append(len(adj[u]) * len(adj[v]))
        scores_aa.append(sum(1.0 / math.log(len(adj[z])) for z in common))
        labels.append(0)
    
    labels = np.array(labels)
    scores_cn = np.array(scores_cn)
    scores_pa = np.array(scores_pa)
    scores_aa = np.array(scores_aa)
    
    metrics = {
        'CN': compute_metrics(scores_cn, labels),
        'PA': compute_metrics(scores_pa, labels),
        'AA': compute_metrics(scores_aa, labels)
    }
    
    return metrics

test_dataset = dataset[split_idx["test"]]

aggregated_metrics = {
    'CN': {'auc': [], 'mrr': [], 'hit5': []},
    'PA': {'auc': [], 'mrr': [], 'hit5': []},
    'AA': {'auc': [], 'mrr': [], 'hit5': []}
}

for data in tqdm(test_dataset):
    metrics = compute_heuristics(data)
    if metrics is not None:
        for method in ['CN', 'PA', 'AA']:
            for key in ['auc', 'mrr', 'hit5']:
                value = metrics[method][key]
                if not np.isnan(value):
                    aggregated_metrics[method][key].append(value)

final_metrics = {}
for method in aggregated_metrics:
    final_metrics[method] = {
        'AUC': np.mean(aggregated_metrics[method]['auc']),
        'MRR': np.mean(aggregated_metrics[method]['mrr']),
        'HIT@5': np.mean(aggregated_metrics[method]['hit5'])
    }

for method in ['CN', 'PA', 'AA']:
    print(f"\n{method} Metrics:")
    print(f"AUC: {final_metrics[method]['AUC']:.4f}")
    print(f"MRR: {final_metrics[method]['MRR']:.4f}")
    print(f"HIT@5: {final_metrics[method]['HIT@5']:.4f}")

100%|██████████| 4113/4113 [00:10<00:00, 387.37it/s]


CN Metrics:
AUC: 0.4129
MRR: 0.0382
HIT@5: 0.0201

PA Metrics:
AUC: 0.6410
MRR: 0.1122
HIT@5: 0.1449

AA Metrics:
AUC: 0.4131
MRR: 0.0385
HIT@5: 0.0205
